In [ ]:
!git clone https://github.com/CryAndRRich/codapath.git

In [ ]:
%cd /kaggle/working/codapath
CODAPATH = "/kaggle/working/codapath"

In [ ]:
import subprocess, sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "huggingface_hub", "hf-transfer"])

In [ ]:
# pathmnist | histoset | skintissue
DATASET = "histoset"

# random | coreset | typiclust | activeft | badge | entropy | margin
# codapath | scalpel | scalpel_multiscale | uncertainty_herding | tcm | dropquery | refine
SAMPLER_NAME = "codapath"

SEED = 42

# Directory holding pre-extracted DINOv2 features (see extract_features.ipynb).
# main() reuses a cached {DATASET}_seed{SEED}_..._train/test.npy if present,
# otherwise it extracts and writes it here. Point this at a mounted Kaggle
# input dataset to skip extraction entirely, e.g.
# FEATURE_DIR = "/kaggle/input/nckh2026-features/features"
FEATURE_DIR = "features"

# Optional: override the config's shared cumulative_budget for a scoped
# experiment run (e.g. SCALPEL-Multiscale on PathMNIST) without touching
# config.yaml's global default used by every other sampler/experiment.
# e.g. CUMULATIVE_BUDGET_OVERRIDE = [25, 50, 75, 100, 125, 150, 175, 200]
CUMULATIVE_BUDGET_OVERRIDE = None

In [ ]:
import os
from huggingface_hub import snapshot_download, login

login("YOUR_HUGGINGFACE_TOKEN")
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

# All samplers share the frozen DINOv2 backbone — it is the only model needed.
print("Downloading facebook/dinov2-base...")
snapshot_download(repo_id="facebook/dinov2-base")

In [ ]:
import sys

os.environ["TOKENIZERS_PARALLELISM"] = "false"
if CODAPATH not in sys.path:
    sys.path.append(CODAPATH)

In [ ]:
import yaml
import torch

from run import main

In [ ]:
PATHMNIST_PATH  = "/kaggle/input/datasets/cryandrrich/nckh2026/pathmnist_224.npz"
HISTOSET_PATH   = "/kaggle/input/datasets/cryandrrich/nckh2026/HistoSet-5x14/HistoSet-5x14"
SKINTISSUE_PATH = "/kaggle/input/datasets/cryandrrich/nckh2026/SkinTissue/SkinTissue/tiles"

DATA_DICT = {
    "pathmnist":  PATHMNIST_PATH,
    "histoset":   HISTOSET_PATH,
    "skintissue": SKINTISSUE_PATH,
}

In [ ]:
CONFIG_PATH = "config/config.yaml"

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

training_cfg = config.get("training", {})
dataset_info = config["datasets"][DATASET]
sampler_cfg  = config.get("samplers", {}).get(SAMPLER_NAME, {})

# Override sampler hyperparams from the notebook (no repo/config edit needed).
# e.g. ablation:  SAMPLER_OVERRIDES = {"use_stain_discount": False}
SAMPLER_OVERRIDES = {}
sampler_cfg = {**sampler_cfg, **SAMPLER_OVERRIDES}
print(f"[{SAMPLER_NAME}] sampler_cfg =", sampler_cfg)

In [ ]:
main(
    data_path=DATA_DICT[DATASET],
    sampler_name=SAMPLER_NAME,
    num_classes=dataset_info["num_classes"],
    cumulative_budget=CUMULATIVE_BUDGET_OVERRIDE or config["cumulative_budget"],
    data_descriptions=dataset_info["descriptions"],
    prompt_templates=config["prompt_templates"],
    sampler_cfg=sampler_cfg,
    probe_epochs=training_cfg["probe_epochs"],
    probe_lr=training_cfg["probe_lr"],
    device=torch.device(config["device"]),
    random_seed=SEED,
    save_dir=f"checkpoints/{DATASET}",
    verbose=True,
    model_cfg=config.get("models", {}),
    feature_cache_dir=FEATURE_DIR,
)